# Лабораторна #1: Класифікація текстів

У цьому завданні ми натренуємо модель для визначення **тональності тексту**. Наша модель отримуватиме на вхід відгук на фільм й видаватиме один з двох класів: позитивний (фільм сподобався) чи негативний (фільм не сподобався).

# 1 Дані

Існує багато датасетів для класифікації текстів. Зручно переглянути їх можна на сервісі [Hugging Face Datasets](https://huggingface.co/datasets?task_categories=task_categories:text-classification).

Для цієї лабораторної ми візьмемо один з класичних датасетів аналізу тональності тексту, [IMDB](https://huggingface.co/datasets/stanfordnlp/imdb). Цей датасет містить 50,000 позначених відгуків на фільми та додатковий непозначений розділ. Кожен позначений відгук відноситься до одного з двох класів:
* позитивний (фільм сподобався);
* негативний (фільм не сподобався).

Для тренування використовується лише 25,000 відгуків, решта 25,000 призначена для оцінки моделі.


Завантажімо дані за допомогою бібліотеки Hugging Face [Datasets](https://huggingface.co/docs/datasets/loading):

In [ ]:
!pip install --quiet datasets spacy pystemmer matplotlib "scikit-learn>=1.8,<2"

In [ ]:
import datasets

In [ ]:
# Метод `load_dataset` завантажує потрібні дані з інтернету та кешує їх
# локально. Повторний виклик цього коду перевикористає вже завантажені
# дані.
imdb = datasets.load_dataset("stanfordnlp/imdb")

`imdb`, як і будь-який інший датасет бібліотеки `datasets`, це словник.
Ключами в ньому є назви розділів (`train`, `test`, `unsupervised`), а значеннями --
об'єкт `Dataset`, який вже містить дані:

In [ ]:
imdb

In [ ]:
print(imdb.keys())
train_data = imdb["train"]
train_data

In [ ]:
# Можемо трактувати `train_data` як масив записів:
train_data[100]["text"]

In [ ]:
# label=0 -- негативний відгук
# label=1 -- позитивний відгук
train_data[100]["label"]

# 2 Підготовка тексту (text preprocessing)

Спочатку ми маємо підготувати текст до подачі в модель класифікації.

### 2.1 Токенізація

Існує багато бібліотек для токенізації. [SpaCy](https://spacy.io/) та [NLTK](https://www.nltk.org/) -- це два великих пакети, що містять в собі багато інструментів для опрацювання текстів, тому числі й токенізатори.

Наступна функція токенізує текст за допомогою spaCy:

In [ ]:
!pip install --quiet spacy

In [ ]:
import spacy
import tqdm
from typing import List, Tuple

# Spacy підтримує багато мов. Зараз нам потрібна англійська
spacy_nlp = spacy.blank("en")


def tokenize_spacy(text: str) -> List[str]:
  """Tokenize string with SpaCy. """

  tokens = spacy_nlp.tokenizer(text)
  return [str(token) for token in tokens]


def tokenize(text: str) -> List[str]:
  return tokenize_spacy(text)


tokenize_spacy("I can't find my $25!  http://google.com/ oleksiy.syvokon@gmail.com ")

Свій токенізатор можна побудувати на регулярних виразах. Найпростіший, хоча й не досконалий, виглядає так:

In [ ]:
import re

def tokenize_re(text: str) -> List[str]:
  return re.findall(r"\w+", text)

tokenize_re("I can't find my $25! :)  http://google.com/ oleksiy.syvokon@gmail.com ")

Такий простий токенізатор далекий від ідеалу. Він пропускає пунктуацію,
неправильно розбиває слова з апосторофом, не знає про існування URL та emoji тощо.

Для того, щоб покрити більше випадків, доведеться ускладнювати regexp.
Ось реальний приклад, наскільки складним це може стати: [`nltk.TweeterTokenizer`](https://github.com/nltk/nltk/blob/637af5380d6071517a5f0d224649e5c3560b5f91/nltk/tokenize/casual.py#L118)

### 2.2 Стемінг

In [ ]:
!pip install pystemmer

In [ ]:
import Stemmer
from typing import List

def stem(tokens: List[str]) -> List[str]:
  """Lower-case and stem tokens. """

  stemmer = Stemmer.Stemmer("english")
  tokens = [tok.lower() for tok in tokens]
  return stemmer.stemWords(tokens)

example_tokens = tokenize(train_data[2]["text"])
stemmed_tokens = stem(example_tokens)
print(" ".join(stemmed_tokens))


In [ ]:
# Знайдемо стеми деяких слів
for word in ("sing", "sings", "singing", "singer", "singers"):
  stemmed = stem([word])[0]
  print(f"{word:<20}   =>   {stemmed}")


Токенізуємо та стемінгуємо текст, щоб оцінити вплив стемінгу на розмір словника.

In [ ]:
from tqdm import tqdm

all_tokens = []
all_tokens_stemmed = []
for doc in tqdm(train_data):
  doc_tokens = tokenize(doc["text"])
  all_tokens += doc_tokens
  all_tokens_stemmed += stem(doc_tokens)

print("Original unique tokens: {:,}".format(len(set(all_tokens))))
print("Stemmed  unique tokens: {:,}".format(len(set(all_tokens_stemmed))))

Як бачимо, кількість унікальних токенів значно зменшилася після стемінгу.

### 2.3 Стоп-слова


Деякі слова зустрічаються часто, але не несуть смислового навантаження. Їх можна видалити з тексту.

Існують готові спискі стоп-слів. Їх можна брати за основу, але варто переглядати
під кожну конкретну задачу. Наприклад, для нашої задачі аналізу тональності
тексту, слова "not", "don't", "like" є важливими, хоча вони присутні в більшості списків стоп-слів.

Зараз ми вважатимемо кілька частотних слів стоп-словами:

In [ ]:
STOP_WORDS = stem(["the", "and", "a", "of", "to", "is", "in", "that", "this", "was", "as", "with", "for", "you", "are", "it"])

def remove_stop_words(tokens: List[str]) -> List[str]:
  return [token for token in tokens if token not in STOP_WORDS]

remove_stop_words("and this was fantastic".split())

### 2.4 Збираємо докупи

Наш невеличкий ланцюжок перетворень складається з трьох кроків: токенізація, стемінг та видалення стоп-слів. Ці три функції мусимо викликати для будь-якого вхідного тексту. Якщо забути зробити один з цих кроків, то модель продовжить працювати, але з гіршою якістю.

Хорошею практикою є збирання усіх кроків в одну функцію. В бібліотеках це називається preprocessing pipeline або pipes.

In [ ]:
def preprocess(text: str) -> List[str]:
  tokens = tokenize(text)
  tokens = stem(tokens)
  tokens = remove_stop_words(tokens)
  return tokens

preprocess("I likes it!")

### 2.5 Словник

Майже завжди в моделях NLP ми маємо справу з обмеженим набором слів (правильніше казати "токенів", бо до них також відносяться пунктуація, n-грами та спеціальні символи). Цей набір токенів називається словником (vocabulary).

Перед тим, як розпочати тренування моделі, ми маємо визначитися, якого розміру буде наш словник та що в нього включати. Великий словник означає, що наша модель матиме багато ознак (features), що підвищує ризик перенавчання (overfitting). Маленький словник призведе до того, що багато вхідних слів, які не потрапили до словника, будуть просто пропущені моделю і можлива ситуація недонавчання (underfit) -- модель стає надто простою для адекватного моделювання задачі.

Простих рецептів по вибору розміру словника немає. В контексті глибинного навчання зазвичай використовується від 8,000 до 64,000 токенів. Для простих лінійних класифікаторов типу того, що ми його зараз будуємо, доречними будуть значення в районі кількох тисяч. Можна вважати розмір словника гіперпараметром та підібрати оптимальне значення кросвалідацією. Поки що зупинимося на розмірі в $N=5000$ токенів.

Як правило, в словник включають начастотніші $N$ токенів, які зустрілися в тренувальних даних. Зі словника можуть виключати пунктуацію та деякі стоп-слова, якщо вони не є корисними для конкретної задачі. Також в словник додають спеціальні токени, які позначають невідомі слова (`<UNK>`), початок та кінець речення (`<s>` та `</s>` або `<bos>`/`</eos>`).

#### 2.5.1 Розподіл слів

Подивимося на розподіл частот слів:

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Корпус вже токенізовано в all_tokens та all_tokens_stemmed
assert all_tokens
assert all_tokens_stemmed

# Злічимо усі токени
token_counts = Counter(all_tokens_stemmed)

# Подивимося на 50 найчастотніших токенів
most_common = token_counts.most_common(50)

most_common_count = sum(count for token, count in most_common)
all_count = sum(token_counts.values())
print(f"Top {len(most_common)} most common tokens account for {most_common_count / all_count:%} of tokens")
print(f"Top {len(most_common)} tokens are: {most_common}")

# Розподіл токенів за частотою
plt.yscale("log")
plt.plot(sorted(token_counts.values(), reverse=True))


Щоб краще уявити собі нерівномірність цього розподілу, намалюємо його на лінійній, а не логаріфмічній шкалі:

In [ ]:
plt.plot(sorted(token_counts.values(), reverse=True))

#### 2.5.2 Рідковживані слова

Залишимо лише ті слова, які зустрічаються щонайменше 5 разів. Також візьмемо `VOCAB_SIZE` найчастотніших слів.

In [ ]:
VOCAB_SIZE = 5000
MIN_COUNT = 5

print(f"Vocab size before filtering: {len(token_counts):,}")

vocab_list = [token for token, count in token_counts.most_common()
              if count >= MIN_COUNT
              and token not in STOP_WORDS]

print(f"Vocab size after filtering: {len(vocab_list):,}")

# Зараз vocab містить токени, відсортовані за частотою у порядку зменшення.
# Візьмемо лише частину з них
vocab_list = vocab_list[:VOCAB_SIZE]
print(f"Final vocab size: {len(vocab_list):,}")

In [ ]:
# Приклад токенів у словнику
vocab_list[4090:4100]

#### 2.5.3 Допоміжні функції

Є кілька операцій, які часто доводиться робити із словником:

- Отримати слово за його індексом
- Отримати індекс за словом

До того ж доводиться враховувати, що токена може й не бути в словнику. У такому випадку зазвичай повертається спеціальний токен `<unk>`.

Нижче ми загорнемо всі ці допоміжні функції у клас `Vocabulary`. Різні бібліотеки і фреймворки мають свої аналоги цього класу.

In [ ]:
class Vocabulary:

  def __init__(self, tokens, unk_token="<unk>"):
    self.unk_token = unk_token
    self.unk_index = 0
    self._itos = [unk_token] + tokens
    self._stoi = {token: index for index, token in enumerate(self._itos)}

  def stoi(self, token: str) -> int:
    """Return token index or `<unk>` index if `token` is not in the vocab.
    """
    return self._stoi.get(token, self.unk_index)


  def itos(self, index: int) -> str:
    """Return token by its `index`.

    Raise LookupError if `index` is out of vocabulary range.
    """

    return self._itos[index]

  @property
  def tokens(self):
    return self._itos

  def __len__(self) -> int:
    return len(self._itos)


vocab = Vocabulary(vocab_list)
assert vocab.stoi("definitely-not-in-vocabulary") == vocab.unk_index
assert vocab.itos(vocab.stoi("dog")) == "dog"
print(f"Index of `dog` is {vocab.stoi('dog')}")
print(f"Token #42 is `{vocab.itos(42)}`")

Токени, яких немає в словнику, предсталяються спеціальним токеном `<unk>` (має індекс 0):

In [ ]:
s = "Hello NULP !".split()
[vocab.stoi(x.lower()) for x in s]

# 3 Представлення тексту

## 3.1 One-hot

При one-hot кодуванні кожне слово представляється вектором, довжина якого дорівнює розміру словника. У цьому векторі всі елементи, крім одного, дорівнюють нулю. Єдина одиниця відповідає позиції слова у словнику.


In [ ]:
def one_hot(token: str, vocab: Vocabulary) -> List[int]:
    # Створюємо список нулів довжиною, що дорівнює розміру словника
    result = [0] * len(vocab)

    # Отримуємо індекс токена (слова) у словнику
    index = vocab.stoi(token)

    # Встановлюємо 1 на позиції, що відповідає індексу токена
    result[index] = 1

    return result

_test_vocab = Vocabulary(["cat", "dog"])
_test_vector = one_hot("dog", _test_vocab)
assert len(_test_vector) == len(_test_vocab)
assert sum(_test_vector) == 1
assert _test_vector[_test_vocab.stoi("dog")] == 1

In [ ]:
# Закодуємо слово "dog"
vector_dog = one_hot("dog", vocab)

# Перевіряємо довжину вектора (вона повинна дорівнювати розміру словника)
len(vector_dog)

In [ ]:
# Виводимо перші 20 елементів вектора. Вектор розріджений, містить багато нулів.
vector_dog[:20]

In [ ]:
# Знаходимо позицію слова "dog" у словнику
vector_dog.index(1)

In [ ]:
# Перевіряємо, чи правильно закодовано слово, декодуючи його індекс назад у слово
vocab.itos(vector_dog.index(1))

## 3.2 Bag-of-words

In [ ]:
def bag_of_words(tokens: List[str], vocab: Vocabulary) -> List[int]:
  result = [0] * len(vocab)

  for token in tokens:
    index = vocab.stoi(token)
    result[index] += 1

  return result

_test_vocab = Vocabulary(["like", "turtle"])
_test_bow = bag_of_words(["like", "like", "turtle"], _test_vocab)
assert _test_bow[_test_vocab.stoi("like")] == 2
assert sum(_test_bow) == 3

In [ ]:
tokens = preprocess("I like like turtles . . . ")
print(f"Tokenized: {tokens}")
vector_text = bag_of_words(tokens, vocab)
print(vector_text)

In [ ]:
bag_of_words(preprocess("turtles . . . like I like"), vocab)[:20]

## 3.3 N-grams


"don't like" -> do, n't, like

I like it very much

1-ngram: I, like, it, very, much

2-ngram: "I like", "like it"

3-ngram: "I like it", "like it very", "it very much"





In [ ]:
def extract_ngrams(tokens: List[str], max_n: int, min_n: int = 1) -> List[str]:
    """Extract n-grams from `s`.

    Args:
        tokens: list of tokens.
        min_n: Minimum length of n-gram to extract.
        max_n: Maximum length of n-gram to extract.

    Return:
        list of ngrams, where each ngram is as a string.


    Example:
        >>> extract_ngrams("Hello world !".split(), 2)
        ['Hello', 'Hello world', 'world', 'world !', '!']

    """

    return list(extract_ngrams_iter(tokens, max_n, min_n))


def extract_ngrams_iter(tokens: List[str], max_n: int, min_n: int = 1):
    """Extract all ngrams from `s`, yielding them one by one.

    Example:
        >>> next(extract_ngrams_iter("Hello world !".split(), 2))
        'Hello'

    """

    for i in range(len(tokens)):
        for j in range(min_n, max_n + 1):
            if i + j <= len(tokens):
                ngram = tokens[i : i + j]
                ngram = " ".join(ngram)
                yield ngram


_test_ngrams = extract_ngrams("hello world !".split(), 2)
assert _test_ngrams == ["hello", "hello world", "world", "world !", "!"]
_test_ngrams

In [ ]:
# Побудуємо н-грамний словник
counts = Counter()
for doc in tqdm(train_data):
  tokens = preprocess(doc["text"])
  ngrams = extract_ngrams(tokens, 2)
  counts.update(ngrams)

ngrams_vocab = [token for token, count in counts.most_common() if count >= MIN_COUNT]
ngrams_vocab = ngrams_vocab[:VOCAB_SIZE]
ngrams_vocab = Vocabulary(ngrams_vocab)


In [ ]:
len(ngrams_vocab)

In [ ]:
# Подивимося на приклади n-грам в нашому словнику:
[x for x in ngrams_vocab._itos ][1000:1020]

### 3.3.1 Bag-of-n-grams

In [ ]:
def bag_of_ngrams(tokens: List[str], vocab: Vocabulary) -> List[int]:
  result = [0] * len(vocab)
  ngrams = extract_ngrams(tokens, 2)
  for ngram in ngrams:
    index = vocab.stoi(ngram)
    result[index] += 1

  return result

In [ ]:
print(bag_of_ngrams(preprocess("I couldn't resist"), ngrams_vocab))

## 3.4 TF-IDF

Bag-of-words надає велику вагу словам, які часто зустрічаються в конкретному документі. Проте слово, яке трапляється майже в усіх документах, зазвичай менш інформативне. **TF-IDF** враховує обидві обставини:

1. TF (*term frequency*) — частота токена у документі: $\text{tf}(t,d)=\frac{count(t,d)}{|d|}$.
2. IDF (*inverse document frequency*) — обернена документна частота: $\text{idf}(t)=\log\frac{1+N}{1+df(t)}+1$, де $N$ — кількість документів, а $\text{df}(t)$ — кількість документів, у яких є токен $t$.

Остаточна вага: $\text{tfidf}(t,d)=\text{tf}(t,d)\cdot [\text{idf}(t)$. Одиниці у формулі IDF згладжують значення і запобігають діленню на нуль.

In [ ]:
import math

def tf_idf(tokens: List[str], vocab: Vocabulary, document_frequency: Counter, n_documents: int) -> List[float]:
  result = [0.0] * len(vocab)
  if not tokens:
    return result

  token_counts = Counter(tokens)
  for token, count in token_counts.items():
    index = vocab.stoi(token)
    if index == vocab.unk_index:
      continue
    term_frequency = count / len(tokens)
    inverse_document_frequency = math.log((1 + n_documents) / (1 + document_frequency[token])) + 1
    result[index] = term_frequency * inverse_document_frequency

  return result

# Невеликий приклад, який легко перевірити вручну.
toy_documents = [
    ["good", "movie", "good"],
    ["bad", "movie"],
]
toy_vocab = Vocabulary(["good", "bad", "movie"])
toy_document_frequency = Counter()
for document in toy_documents:
  toy_document_frequency.update(set(document))

# TF-IDF для першого документу
toy_tfidf = tf_idf(toy_documents[0], toy_vocab, toy_document_frequency, len(toy_documents))
assert toy_tfidf[toy_vocab.stoi("good")] > toy_tfidf[toy_vocab.stoi("movie")]
assert toy_tfidf[toy_vocab.unk_index] == 0.0
{token: round(toy_tfidf[toy_vocab.stoi(token)], 3) for token in toy_vocab.tokens[1:]}

NameError: name 'List' is not defined

## Feature hashing

Feature hashing, також відомий як hashing trick, це альтернатива створенню повного словника всіх слів. Цей метод використовує хеш-функцію, щоб перетворити слова на числові ознаки в заданному діапазоні (наприклад, від 0 до 5000). Хеш-функція без проблем приймає будь-які токени, знайомі та незнайомі. Більше того, словник тепер можна не зберігати. Такий підхід економить пам'ять і прискорює обробку даних.

Однак у feature hashing є свої недоліки. Головний з них - можливість колізій, коли різні слова потрапляють в один індекс вектора. Крім того, отримані вектори важко інтерпретувати, оскільки кожен індекс не відповідає конкретному слову.

Загалом, feature hashing зараз використовується рідко. Переважно, у випадках, коли потрібен абсолютно мінімальний розмір моделі та швидкість. Наприклад, модель визначення мови веб-сторінки, вбудована в Google Chrome, важила менше одного мегабайта й була здатна визначати близько ста мов завдяки трюку з хешуванням ознак.

In [ ]:
from hashlib import md5

def bag_of_hashed_ngrams(ngrams: List[str], vocab_size) -> List[int]:
    result = [0] * vocab_size
    for ngram in ngrams:
        ngram_hash = md5(ngram.encode()).digest()
        index = int.from_bytes(ngram_hash, byteorder='big') % vocab_size
        result[index] += 1
    return result

_test_hashed = bag_of_hashed_ngrams(["hello", "hello", "world"], vocab_size=10)
assert len(_test_hashed) == 10
assert sum(_test_hashed) == 3

In [ ]:
bag_of_hashed_ngrams(["hello", "world"], vocab_size=10)

# 4 Класифікація тексту

## Векторизація даних

In [ ]:
import numpy as np


def vectorize(data: 'datasets.Dataset') -> Tuple[np.array, np.array]:
  X_list = []
  y_list = []
  for doc in data:
    tokens = preprocess(doc["text"])
    X_list.append(bag_of_ngrams(tokens, ngrams_vocab))
    y_list.append(doc["label"])  # 0 or 1
  X = np.array(X_list)
  y = np.array(y_list)
  return (X, y)

# Take random 2000 documents for a dev set
valid_data = imdb["test"].shuffle(seed=1).filter(lambda x, i: i < 2000, with_indices=True)

X_train, y_train = vectorize(train_data)
X_val, y_val = vectorize(valid_data)




In [ ]:
X_train.shape
X_val

## Тренування моделі

In [ ]:
%%time
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=0.5, l1_ratio=1)
model.fit(X_train, y_train)


## Оцінка моделі

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def evaluate(model):
  y_hat = model.predict(X_val)
  accuracy = (y_hat == y_val).mean()

  return {
      "accuracy": accuracy,
      "precision": precision_score(y_val, y_hat),
      "recall": recall_score(y_val, y_hat),
      "f1": f1_score(y_val, y_hat),
  }

metrics = evaluate(model)
print(f"Log regression: {metrics}")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
import matplotlib.pyplot as plt

y_score = model.predict_proba(X_val)[:, 1]
PrecisionRecallDisplay.from_predictions(
    y_val,
    y_score,
    name="Logistic regression",
)
plt.title("Precision–Recall curve")
plt.show()

In [ ]:
def predict(text: str, model: 'BaseEstimator') -> float:
  tokens = preprocess(text)
  features = bag_of_ngrams(tokens, ngrams_vocab)
  features = np.array([features])
  prob_neg, prob_pos = model.predict_proba(features)[0]
  return prob_pos

In [ ]:
threshold = 0.5
predict("I liked that movie!", model) > threshold

In [ ]:
negative_probability = predict("The movie is terrible ", model)
assert 0.0 <= negative_probability <= 1.0
negative_probability > threshold

# 5 scikit-learn

До цього моменту ми власноруч писали функції для препроцесінгу даних. Це добре для навчання, але в реальності використовують готові імплементації.

Наприклад, клас [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) з пакету scikit-learn вміє робити майже все з написаного нами:

In [ ]:
import datasets
dataset = datasets.load_dataset('stanfordnlp/imdb')
train_data = dataset['train']
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(
    max_features=1000,
    ngram_range=(1,3),
    binary=False,
    stop_words=STOP_WORDS,
    # vocabulary=vocab.tokens,
    )
X_train = vectorizer.fit_transform(train_data[:]["text"])

In [ ]:
X_train

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
%%time
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression(solver='liblinear', C=0.1, l1_ratio=1)
model2.fit(X_train, train_data["label"])

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def classification_metrics(y_true, y_pred):
  return {
      "accuracy": accuracy_score(y_true, y_pred),
      "precision": precision_score(y_true, y_pred),
      "recall": recall_score(y_true, y_pred),
      "f1": f1_score(y_true, y_pred),
  }

valid_data = dataset["test"].shuffle(seed=1).filter(lambda x, i: i < 2000, with_indices=True)
X_valid = vectorizer.transform(valid_data["text"])
y_valid = valid_data["label"]

y_pred = model2.predict(X_valid)
classification_metrics(y_valid, y_pred)

## TF-IDF with sklearn

In [ ]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    # binary=True,
    # stop_words='english',
    # vocabulary=vocab.tokens,
    )
X_train = vectorizer.fit_transform(train_data[:]["text"])

model3 = LogisticRegression(solver='liblinear', C=0.2, l1_ratio=1)
model3.fit(X_train, train_data["label"])


X_valid = vectorizer.transform(valid_data["text"])
y_valid = valid_data["label"]
y_pred = model3.predict(X_valid)
classification_metrics(y_valid, y_pred)

## Завдання: порівняйте представлення тексту

Запустіть кілька варіантів моделі за однакового розбиття даних і заповніть таблицю.

| Модель | Ознаки | `max_features` | `ngram_range` | Accuracy | Precision | Recall | F1 | Час навчання |
|---|---|---:|:---:|---:|---:|---:|---:|---:|
| Logistic regression | CountVectorizer | 1000 | (1, 2) | — | — | — | — | — |
| Logistic regression | CountVectorizer | 5000 | (1, 2) | — | — | — | — | — |
| Logistic regression | CountVectorizer | 10000 | (1, 2) | — | — | — | — | — |
| Logistic regression | CountVectorizer | 20000 | (1, 2) | — | — | — | — | — |
| Logistic regression | TF-IDF | 10000 | (1, 2) | — | — | — | — | — |
| Logistic regression | TF-IDF | 10000 | (1, 1) | — | — | — | — | — |
| Logistic regression | TF-IDF | 10000 | (1, 3) | — | — | — | — | — |
| Ваш варіант | … | … | … | — | — | — | — | — |

Коротко підсумуйте результати експериментів. Як кількість ознак впливає на якість? Як вплинуло додавання біграм та триграм? Який результат дали TF-IDF ознаки порівнянно з простою кількістю зустрічань?